# STEP 1 : Dataset Ingestion & Validation

**To initiate the pipeline, we must first load  selected snehaanbhawal/resume-dataset. This specific dataset typically contains unstructured data organized under columns named Category and Resume_str (or Resume_text). To ensure system stability, we will implement a robust data loader designed to standardize these column schemas, handle missing or null entries, and prevent downstream parsing failures.**

In [3]:
import pandas as pd
import numpy as np
import os

def load_and_validate_dataset(filepath: str) -> pd.DataFrame:
    """
    Loads the Kaggle resume dataset, standardizes schema columns,
    and returns a clean dataframe with validated non-null records.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Could not find '{filepath}'. Please check your working directory.")
        
    # Read the raw CSV file
    df = pd.read_csv(filepath)
    print(f"Original Dataset Shape: {df.shape}")
    print(f"Detected Headers: {list(df.columns)}")
    
    # Map the detected headers to standard internal pipeline variable names
    if 'Resume_str' in df.columns:
        df = df.rename(columns={'Resume_str': 'resume_text'})
    elif 'Resume_html' in df.columns and 'resume_text' not in df.columns:
        df = df.rename(columns={'Resume_html': 'resume_text'})
        print("Notice: 'Resume_str' missing. Defaulting to 'Resume_html' source extraction.")
    
    if 'Category' in df.columns:
        df = df.rename(columns={'Category': 'category'})
        
    # Drop records that contain null value definitions inside the resume body
    df = df.dropna(subset=['resume_text']).reset_index(drop=True)
    df['category'] = df['category'].fillna('Unknown').str.strip()
    
    print(f"Data validation complete. Active Records for Pipeline: {len(df)}")
    return df

# Local Execution Check
# Update 'Resume.csv' if your unzipped file has a different name in the directory
try:
    resume_df = load_and_validate_dataset('Resume.csv')
    print("\nFirst 3 rows preview:")
    print(resume_df[['category', 'resume_text']].head(3))
except Exception as error_log:
    print(f"Execution Error: {str(error_log)}")

Original Dataset Shape: (2484, 4)
Detected Headers: ['ID', 'Resume_str', 'Resume_html', 'Category']
Data validation complete. Active Records for Pipeline: 2484

First 3 rows preview:
  category                                        resume_text
0       HR           HR ADMINISTRATOR/MARKETING ASSOCIATE\...
1       HR           HR SPECIALIST, US HR OPERATIONS      ...
2       HR           HR DIRECTOR       Summary      Over 2...


# Step 2: HTML-Aware Advanced Text Normalization & Cleaning Pipeline.

**Since this specific dataset contains raw HTML strings (Resume_html), this cell will strip out structural HTML markers, remove web artifacts (like &nbsp;), clean out links/emails/punctuation, and drop generic English stop words.**

In [5]:
import re
import nltk
from nltk.corpus import stopwords

# Ensure NLTK resources are available in the runtime environment
try:
    nltk.data.find('corpora/stopwords')
except KeyError:
    nltk.download('stopwords')

def professional_html_text_cleaner(text: str) -> str:
    """
    Strips structural HTML markups, filters special characters, 
    and removes standard non-informative english stopwords.
    """
    if not isinstance(text, str):
        return ""
    
    # Lowercase text conversion
    text = text.lower()
    
    # Strip structural HTML tags/brackets (e.g., <div>, <p>)
    text = re.sub(r'<[^>]*>', ' ', text)
    
    # Strip HTML web entity code (e.g., &nbsp;, &amp;, &lt;)
    text = re.sub(r'&\w+;', ' ', text)
    
    # Remove URLs and email addresses
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', ' ', text)
    
    # Filter out punctuation, special symbols, and numeric structures
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # Normalize structural whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Apply tokenized stopword exclusions
    stop_words = set(stopwords.words('english'))
    cleaned_tokens = [word for word in text.split() if word not in stop_words]
    
    return " ".join(cleaned_tokens)

# Run transformation over data rows and create a new column
if 'resume_df' in locals():
    resume_df['cleaned_resume_text'] = resume_df['resume_text'].apply(professional_html_text_cleaner)
    print("Step 2 processing complete. Parsed text saved under 'cleaned_resume_text'.")
    print("\nCleaned Text Preview (First Row):")
    print(resume_df['cleaned_resume_text'].iloc[0][:300])
else:
    print("Error: 'resume_df' variable is missing from the workspace scope.")

Step 2 processing complete. Parsed text saved under 'cleaned_resume_text'.

Cleaned Text Preview (First Row):
hr administrator marketing associate hr administrator summary dedicated customer service manager years experience hospitality customer service management respected builder leader customer focused teams strives instill shared enthusiastic commitment customer service highlights focused customer satisf


# Step 3: Natural Language Processing Skill Extraction Engine.

**This block loads the spaCy tokenization architecture to process the dataset. It maps a dedicated lookup dictionary (SKILLS_CATALOG) to find specific competencies while enforcing strict token word boundaries (ensuring multi-word skills or shorter matching strings are evaluated accurately without false-positive overlaps).**

In [6]:
import spacy

# Load spaCy English model pipeline
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

def extract_skills(text: str, skill_dictionary: list) -> set:
    """
    Tokenizes text using spaCy and matches keyword arrays on strict word boundaries.
    """
    doc = nlp(text.lower())
    individual_tokens = {token.text for token in doc}
    extracted_skills = set()
    
    for skill in skill_dictionary:
        skill_lower = skill.lower()
        # Evaluate multi-word phrases vs single token word matches
        if " " in skill_lower:
            if skill_lower in text.lower():
                extracted_skills.add(skill)
        else:
            if skill_lower in individual_tokens:
                extracted_skills.add(skill)
                
    return extracted_skills

# Global dictionary definitions matching standard cross-industry competencies
SKILLS_CATALOG = [
    'Python', 'Machine Learning', 'SQL', 'NLP', 'Tableau', 'Java', 
    'JavaScript', 'HTML5', 'Customer Service', 'Management', 'Marketing'
]

# Runing localized extraction test on the first row of cleaned data
if 'resume_df' in locals() and 'cleaned_resume_text' in resume_df.columns:
    test_extraction = extract_skills(resume_df['cleaned_resume_text'].iloc[0], SKILLS_CATALOG)
    print("Step 3 processing complete.")
    print(f"Skills extracted from Row 0: {list(test_extraction)}")
else:
    print("Error: DataFrame or 'cleaned_resume_text' column from Step 2 is missing.")

Step 3 processing complete.
Skills extracted from Row 0: ['Customer Service', 'Marketing', 'Management']


# Step 4: Representation Learning & Similarity Scoring (TF-IDF + Cosine Similarity).

**This cell converts the text corpus into numerical representations using a TF-IDF vectorizer. It appends your targeted job description to the 2,484 cleaned resume text matrix, extracts the vectors, and computes the absolute Cosine Similarity metric to give every candidate an objective match score.**

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

def calculate_similarity_scores(job_desc: str, dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Vectorizes the job description and cleaned resumes using TF-IDF,
    computes the cosine similarity metric, and maps scores to the dataframe.
    """
    if 'cleaned_resume_text' not in dataframe.columns:
        raise KeyError("Required column 'cleaned_resume_text' missing. Ensure Step 2 has run completely.")
        
    # Standardize and preprocess the incoming target job description using Step 2's function
    cleaned_jd = professional_html_text_cleaner(job_desc)
    
    # Establish corpus boundaries: Job description sits at index 0, followed by all resumes
    corpus = [cleaned_jd] + dataframe['cleaned_resume_text'].tolist()
    
    # Configure and build TF-IDF weights
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(corpus)
    
    # Extract reference vectors
    jd_vector = tfidf_matrix[0]
    resume_matrix = tfidf_matrix[1:]
    
    # Compute dot product normalized similarity arrays
    similarity_scores = cosine_similarity(jd_vector, resume_matrix).flatten()
    
    # Assign data mapping to a fresh copy of the dataframe
    processed_df = dataframe.copy()
    processed_df['match_score_pct'] = np.round(similarity_scores * 100, 2)
    
    return processed_df

# Definition of evaluation constraints for pipeline testing
target_job_description = """
Looking for a Senior Data Scientist. The ideal candidate must have extensive 
knowledge of Python, Machine Learning, and SQL. Experience building production 
NLP pipelines and Data Visualization dashboards using Tableau is highly desirable.
"""

# Execution checkpoint for your 2484 records
if 'resume_df' in locals():
    scored_df = calculate_similarity_scores(target_job_description, resume_df)
    print("Step 4 processing complete.")
    print(f"Scored DataFrame shape: {scored_df.shape}")
    print("\nPreview of top 5 highest matching rows based on TF-IDF weights:")
    print(scored_df.sort_values(by='match_score_pct', ascending=False)[['category', 'match_score_pct']].head())
else:
    print("Error: 'resume_df' from Step 1 is missing from the workspace scope.")

Step 4 processing complete.
Scored DataFrame shape: (2484, 6)

Preview of top 5 highest matching rows based on TF-IDF weights:
           category  match_score_pct
1218     CONSULTANT            23.54
1762    ENGINEERING            19.17
1303  DIGITAL-MEDIA            18.02
1339     AUTOMOBILE            15.60
1142     CONSULTANT            14.91


# Step 5: Skill Gap & HR Analytics Decision-Support Module.

**This block maps deterministic token overlap logic row-by-row using the extract_skills engine from Step 3. It targets the explicit skills specified in the job description, compares them against the candidate's skills, and isolates both the intersection (skills matched) and the difference (skills missing).**

In [8]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def execute_standalone_analytics(job_desc: str, skill_catalog: list, dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Combines vector similarity scoring and regex skill matching to process 
    analytics natively without relying on outside cell variables.
    """
    if 'cleaned_resume_text' not in dataframe.columns:
        raise KeyError("Required column 'cleaned_resume_text' missing. Run Step 2 first.")
    
    # --- Part A: Compute Similarity Scores (Step 4 Logic) ---
    cleaned_jd = professional_html_text_cleaner(job_desc)
    corpus = [cleaned_jd] + dataframe['cleaned_resume_text'].tolist()
    
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(corpus)
    
    jd_vector = tfidf_matrix[0]
    resume_matrix = tfidf_matrix[1:]
    
    similarity_scores = cosine_similarity(jd_vector, resume_matrix).flatten()
    
    analytics_df = dataframe.copy()
    analytics_df['match_score_pct'] = np.round(similarity_scores * 100, 2)
    
    # --- Part B: Compute Skill Gaps (Step 5 Logic) ---
    def quick_extract(text_content: str) -> set:
        if not isinstance(text_content, str):
            return set()
        text_lower = text_content.lower()
        found_skills = set()
        for skill in skill_catalog:
            skill_lower = skill.lower()
            pattern = r'\b' + re.escape(skill_lower) + r'\b'
            if re.search(pattern, text_lower):
                found_skills.add(skill)
        return found_skills

    required_skills = quick_extract(job_desc)
    matched_skills_list = []
    missing_skills_list = []
    
    for index, row in dataframe.iterrows():
        candidate_skills = quick_extract(row['resume_text'])
        matched = required_skills.intersection(candidate_skills)
        missing = required_skills.difference(candidate_skills)
        
        matched_skills_list.append(list(matched))
        missing_skills_list.append(list(missing))
        
    analytics_df['matched_skills'] = matched_skills_list
    analytics_df['missing_skills'] = missing_skills_list
    
    # Sort by mathematical relevance rank
    analytics_df = analytics_df.sort_values(by='match_score_pct', ascending=False).reset_index(drop=True)
    return analytics_df

# Static execution parameters
target_job_description = """
Looking for a Senior Data Scientist. The ideal candidate must have extensive 
knowledge of Python, Machine Learning, and SQL. Experience building production 
NLP pipelines and Data Visualization dashboards using Tableau is highly desirable.
"""

SKILLS_CATALOG = ['Python', 'Machine Learning', 'SQL', 'NLP', 'Tableau', 'Java', 'JavaScript', 'HTML5']

# Direct ingestion verification from the step 1 dataframe
try:
    analyzed_df = execute_standalone_analytics(target_job_description, SKILLS_CATALOG, resume_df)
    print("Step 5 processing complete.")
    print(f"Analyzed DataFrame shape: {analyzed_df.shape}")
    print("\nPreview of top 3 records with explicit skill gap metrics:")
    print(analyzed_df[['category', 'match_score_pct', 'matched_skills', 'missing_skills']].head(3))
except NameError:
    print("Scope Error: 'resume_df' from Step 1 is not defined in memory. Please run Cell 1 first.")

Step 5 processing complete.
Analyzed DataFrame shape: (2484, 8)

Preview of top 3 records with explicit skill gap metrics:
        category  match_score_pct                            matched_skills  \
0     CONSULTANT            23.54           [Python, Machine Learning, SQL]   
1    ENGINEERING            19.17  [Python, SQL, Machine Learning, Tableau]   
2  DIGITAL-MEDIA            18.02                    [Python, Tableau, SQL]   

            missing_skills  
0           [Tableau, NLP]  
1                    [NLP]  
2  [Machine Learning, NLP]  


# Step 6: System Integration & Batch Reporting.

## The goal of this final cell is to transform raw mathematical matrices and list structures into an executive-ready HR report. It acts as the final presentation layer of your machine learning pipeline.

Here is exactly what this cell does behind the scenes:

**Relevance Slicing (head(top_n))**: Out of all 2,484 records processed by your vectorizer and analytics engine, it slices the top 10 rows. Because Step 5 sorted the DataFrame in descending order by match_score_pct, these are guaranteed to be your highest-matching candidates.

**String Formatting Unification**: In your dataset columns, matched and missing skills are stored as native Python lists (e.g., ['Python', 'SQL']). Printing raw lists in a final table looks messy. This cell uses a .apply(lambda x: ", ".join(x)) operation to convert those arrays into clean, comma-separated strings (e.g., "Python, SQL"). If a candidate matches nothing or has zero gaps, it replaces the empty list with a clean "None" flag.

**Schema Labeling Alignment**: It renames your technical, lower-case DataFrame column headers (category, match_score_pct, etc.) into formal presentation headers (Candidate Category, Match Score (%), Skills Matched, Skills Missing).

**Console Output Generation**: Finally, it pipes the clean slice through to_string(index=True) wrapped inside formatting boundaries (===), rendering a perfectly aligned tabular dashboard directly inside your notebook console without truncating the long skill text strings.

In [9]:
import pandas as pd

def generate_final_hr_report(dataframe: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
    """
    Slices and structures the top N evaluated candidates into a clean,
    professional summary report with clear column headers.
    """
    if 'match_score_pct' not in dataframe.columns or 'matched_skills' not in dataframe.columns:
        raise KeyError("Required analysis columns missing. Please ensure Step 5 executed completely.")
        
    # Isolate relevant reporting parameters
    reporting_headers = [
        'category', 
        'match_score_pct', 
        'matched_skills', 
        'missing_skills'
    ]
    
    # Extract top matching candidates
    report_slice = dataframe[reporting_headers].head(top_n).copy()
    
    # Clean list presentation format for the final console printout
    report_slice['matched_skills'] = report_slice['matched_skills'].apply(lambda x: ", ".join(x) if x else "None")
    report_slice['missing_skills'] = report_slice['missing_skills'].apply(lambda x: ", ".join(x) if x else "None")
    
    # Rename columns for executive review presentation
    report_slice.columns = [
        'Candidate Category', 
        'Match Score (%)', 
        'Skills Matched', 
        'Skills Missing'
    ]
    
    return report_slice

# Execution block
try:
    final_report = generate_final_hr_report(analyzed_df, top_n=10)
    print("\n======================= FINAL CANDIDATE RANKING REPORT =======================")
    print(final_report.to_string(index=True))
    print("==============================================================================")
except NameError:
    print("Error: 'analyzed_df' from Step 5 is missing. Run Cell 5 before running this cell.")


======================= FINAL CANDIDATE RANKING REPORT =======================
       Candidate Category  Match Score (%)                          Skills Matched                 Skills Missing
0              CONSULTANT            23.54           Python, Machine Learning, SQL                   Tableau, NLP
1             ENGINEERING            19.17  Python, SQL, Machine Learning, Tableau                            NLP
2           DIGITAL-MEDIA            18.02                    Python, Tableau, SQL          Machine Learning, NLP
3              AUTOMOBILE            15.60                    Python, Tableau, SQL          Machine Learning, NLP
4              CONSULTANT            14.91                            SQL, Tableau  Machine Learning, Python, NLP
5             AGRICULTURE            13.30                    Python, Tableau, SQL          Machine Learning, NLP
6                   SALES            10.87               Machine Learning, Tableau               Python, SQL, NLP
7  INFOR